# Validation v3 — Features couleur + texture (Route B)

Approche **alternative à CLIP** : extraction de features **dédiées au domaine UV-365 nm** (histogrammes couleur HSV/CIELAB, saturation, LBP, GLCM) + 3 classifieurs comparés (RandomForest, SVM-RBF, GradientBoosting).

**Hypothèse** : CLIP capture la composition d'image. Pour la fluorescence, c'est la **couleur + texture** qui compte. Avec des features adaptées, on devrait passer de 18 % (CLIP) à 35-50 % minimum.

**Pipeline** : 6 cellules, ~10 min runtime.

**Pré-requis** : tu dois avoir le fichier `labels_claude.json` du run v2 + tes 80 photos.

## Cellule 1 — Installation + imports

In [ ]:
!pip install -q scikit-image scikit-learn matplotlib seaborn pillow tqdm xgboost

import json
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from skimage import color
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
print('OK')

## Cellule 2 — Upload labels_claude.json + 80 photos

In [ ]:
from google.colab import files

print('=== Etape 1/2 : Upload labels_claude.json ===')
print('Selectionne le fichier labels_claude.json telecharge depuis le notebook v2')
uploaded = files.upload()
labels_filename = [f for f in uploaded.keys() if f.endswith('.json')][0]
with open(labels_filename) as f:
    labels = json.load(f)
print(f'\n{len(labels)} annotations chargees')
print('Distribution :', Counter(l.get("label") for l in labels.values()))

print('\n=== Etape 2/2 : Upload des photos ===')
print('Selectionne tes 80 photos UV (Ctrl+A pour tout selectionner)')
uploaded = files.upload()
image_dir = Path('/content/uv_images')
image_dir.mkdir(exist_ok=True)
for fname, data in uploaded.items():
    (image_dir / fname).write_bytes(data)
image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')])
print(f'\n{len(image_paths)} photos chargees')

# Verification : tous les filenames existent dans labels ?
missing_labels = [p.name for p in image_paths if p.name not in labels]
missing_photos = [fn for fn in labels if not (image_dir / fn).exists()]
if missing_labels:
    print(f'\nATTENTION: {len(missing_labels)} photos sans annotation : {missing_labels[:3]}...')
if missing_photos:
    print(f'\nATTENTION: {len(missing_photos)} annotations sans photo : {missing_photos[:3]}...')

## Cellule 3 — Extraction de features dédiées domaine UV

Chaque image → vecteur ~90 dimensions :
- **HSV histogram** (32 bins teinte + 8 saturation + 8 valeur) — capte les couleurs de fluorescence
- **CIELAB stats** (mean/std/percentiles sur L, a, b) — perceptuellement uniforme
- **Saturation stats** — distinction voile UV vs fluorescence saturée (CRITIQUE)
- **LBP histogram** (Local Binary Patterns) — texture micro-locale
- **GLCM** (Gray-Level Co-occurrence Matrix) — contraste, homogénéité, énergie, corrélation

In [ ]:
def extract_features(img_path, size=512):
    img = Image.open(img_path).convert('RGB')
    img.thumbnail((size, size))
    arr = np.array(img)
    features = []

    # 1. HSV histogram
    hsv = color.rgb2hsv(arr)
    h_hist, _ = np.histogram(hsv[:,:,0], bins=32, range=(0, 1))
    s_hist, _ = np.histogram(hsv[:,:,1], bins=8, range=(0, 1))
    v_hist, _ = np.histogram(hsv[:,:,2], bins=8, range=(0, 1))
    h_hist = h_hist / max(h_hist.sum(), 1)
    s_hist = s_hist / max(s_hist.sum(), 1)
    v_hist = v_hist / max(v_hist.sum(), 1)
    features.extend(h_hist); features.extend(s_hist); features.extend(v_hist)

    # 2. CIELAB statistics (perceptuellement uniforme)
    lab = color.rgb2lab(arr)
    L, a, b = lab[:,:,0], lab[:,:,1], lab[:,:,2]
    for ch in [L, a, b]:
        features.extend([ch.mean(), ch.std(), np.percentile(ch, 10), np.percentile(ch, 90)])

    # 3. Saturation stats (CRITIQUE pour fluorescence)
    s = hsv[:,:,1]
    features.extend([s.mean(), s.max(), s.std(),
                     np.percentile(s, 50), np.percentile(s, 90), np.percentile(s, 99),
                     (s > 0.5).mean(), (s > 0.7).mean()])

    # 4. LBP texture
    gray = color.rgb2gray(arr)
    gray_u8 = (gray * 255).astype(np.uint8)
    lbp = local_binary_pattern(gray_u8, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0, 10))
    lbp_hist = lbp_hist / max(lbp_hist.sum(), 1)
    features.extend(lbp_hist)

    # 5. GLCM (downsampled pour vitesse)
    g_small = gray_u8[::4, ::4]
    glcm = graycomatrix(g_small, distances=[1], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
        features.extend(graycoprops(glcm, prop).flatten())

    return np.array(features, dtype=np.float32)

feats = []
filenames = []
for img_path in tqdm(image_paths, desc='Extraction features'):
    if img_path.name not in labels:
        continue
    feats.append(extract_features(img_path))
    filenames.append(img_path.name)
X = np.stack(feats)
y_all = np.array([labels[fn]['label'] for fn in filenames])
print(f'\nX shape : {X.shape} (echantillons, dimensions)')
print(f'Classes  : {sorted(set(y_all))}')
print(f'Distribution :')
for cat, n in Counter(y_all).most_common():
    print(f'  {cat:12s} {n:3d}')

## Cellule 4 — Comparaison de 3 classifieurs avec cross-validation 5-fold

In [ ]:
viable = {c for c, n in Counter(y_all).items() if n >= 2 and c not in ('error', 'unknown')}
drop = set(Counter(y_all)) - viable
if drop:
    print(f'Classes ignorees (< 2 echantillons ou error/unknown) : {drop}')
keep = np.array([yi in viable for yi in y_all])
X_use, y_use = X[keep], y_all[keep]
print(f'Echantillons utilises : {len(y_use)}')

n_splits = min(5, min(Counter(y_use).values()))
print(f'CV {n_splits}-fold\n')

models = {
    'RandomForest':     Pipeline([('sc', StandardScaler()), ('m', RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42, n_jobs=-1))]),
    'SVM-RBF':          Pipeline([('sc', StandardScaler()), ('m', SVC(kernel='rbf', class_weight='balanced', C=2.0, gamma='scale', random_state=42))]),
    'GradientBoosting': Pipeline([('sc', StandardScaler()), ('m', GradientBoostingClassifier(n_estimators=300, max_depth=3, random_state=42))]),
}

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = {}
for name, clf in models.items():
    preds = cross_val_predict(clf, X_use, y_use, cv=skf, n_jobs=-1)
    acc = (preds == y_use).mean()
    results[name] = (preds, acc)
    print(f'{name:20s}  accuracy = {acc*100:.1f} %')

best_name = max(results, key=lambda k: results[k][1])
best_preds, best_acc = results[best_name]
print(f'\n=== MEILLEUR : {best_name}  ({best_acc*100:.1f} %) ===')
print(f'\nComparaison vs CLIP zero-shot (notebook v2) :')
print(f'  CLIP zero-shot       18.0 %')
for name, (_, acc) in sorted(results.items(), key=lambda x: -x[1][1]):
    delta = (acc - 0.18) * 100
    sign = '+' if delta >= 0 else ''
    print(f'  {name:20s} {acc*100:5.1f} %   ({sign}{delta:.1f} pts vs CLIP)')

## Cellule 5 — Matrice de confusion + classification report (meilleur modèle)

In [ ]:
labels_unique = sorted(set(y_use))
cm = confusion_matrix(y_use, best_preds, labels=labels_unique)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_unique, yticklabels=labels_unique)
plt.xlabel('Predit'); plt.ylabel('Verite (Claude)')
plt.title(f'Matrice de confusion - {best_name} sur features couleur+texture')
plt.tight_layout(); plt.show()

print(classification_report(y_use, best_preds, zero_division=0))

## Cellule 6 — Verdict + interprétation

**Lecture des seuils** :
- **Accuracy < 25 %** : encore au-dessus du hasard mais insuffisant. Domaine UV trop complexe pour 80 photos quel que soit le modèle. → **Plus de données obligatoire**.
- **Accuracy 25-40 %** : signal réel détecté ! Confirme que features couleur+texture > CLIP. → Continue à scaler le dataset.
- **Accuracy 40-55 %** : très bon résultat avec 80 photos. La piste est validée. → Plan de collecte vers 500 photos pour atteindre 70 %.
- **Accuracy > 55 %** : exceptionnel. → On peut viser un modèle de production directement après scaling à 300-500 photos.

**Si on dépasse CLIP de >10 points** → cette approche est la bonne. Le notebook prouve que **features dédiées au domaine** battent CLIP zero-shot.

In [ ]:
print('=== VERDICT FINAL ===\n')
print(f'Meilleur classifieur : {best_name}')
print(f'Accuracy : {best_acc*100:.1f} %')
print()
if best_acc > 0.55:
    print('EXCELLENT - signal tres fort.')
    print('Recommandation : passer a 300-500 photos puis modele de production.')
elif best_acc > 0.40:
    print('TRES BON - signal solide avec 80 photos.')
    print('Recommandation : viser 500 photos balanced. Re-entrainement attendu a 65-75 %.')
elif best_acc > 0.25:
    print('SIGNAL DETECTE - features couleur/texture utiles.')
    print('Recommandation : continuer collecte vers 300 photos pour tester scaling.')
else:
    print('SIGNAL TROP FAIBLE.')
    print('Recommandation : revoir annotations OU collecter beaucoup plus de donnees.')

print('\n=== CLIP zero-shot vs Features couleur+texture ===')
print(f'  CLIP             : 18.0 %')
print(f'  {best_name:16s} : {best_acc*100:.1f} %')
delta = (best_acc - 0.18) * 100
print(f'  Delta            : {"+" if delta >=0 else ""}{delta:.1f} pts')